# Milestone 2 — LaTeX Processing

## Objectives

This milestone focuses on parsing LaTeX source files,
identifying document structure, and extracting relevant elements.

# Part II — Milestone 2: Hierarchical Parsing and Standardization

## 6. Parsing methodology

### 6.1 Main compilation path

The publication root is the `.tex` file containing `\\documentclass`. Follow `\\input` and `\\include` recursively, ignoring commands commented out in LaTeX comments. Parse only files reachable from the root; ignore redundant auxiliary `.tex` files.

Explain how the implementation handles:
- Root detection and multiple possible roots
- Relative paths and omitted `.tex` extensions
- Recursive includes and include cycles
- Comments and missing included files
- Versions without TeX sources (record as empty, do not silently skip)

**Implementation:** TODO

In [ ]:
# TODO: Link or import the actual parser implementation from src/latex_parser/.
# Example:
# from src.latex_parser import find_root_tex, collect_compilation_files, parse_publication

### 6.2 Hierarchy and element extraction

Describe how the parser represents a document as a tree and assigns parent-child relationships. Required leaf granularity includes:
- Sentences from prose, without splitting at non-sentence periods in math, commands, abbreviations (e.g. `i.e.`, `et al.`, `Fig. 1`), URLs, or decimal numbers
- Block formulas (e.g. `equation` environments or `$$...$$`)
- Figures; tables are treated as a type of figure
- Itemized/enumerated lists as branching components with each item at the next level

Include acknowledgements and appendices, including unnumbered `\\section*` sections. Do not parse the references section into the hierarchy.

**Element kinds and hierarchy-level rules:** TODO

**Sentence boundary handling:** TODO

**Formula / figure / table handling:** TODO

### 6.3 Cleanup and math normalization

Explain how non-semantic LaTeX formatting is removed (examples in the specification include `\\centering`, placement options such as `[htpb]`, and `\\midrule`) while preserving semantic content. State the chosen unified representation for inline and block math.

**Cleanup rules:** TODO

**Inline math format:** TODO

**Block math format:** TODO

**Validation checks:** TODO

### 6.4 `\\bibitem` conversion and reference deduplication

Describe the programmatic conversion from `\\bibitem` definitions to standard BibTeX entries. Then explain cross-version deduplication:
- Identify entries referring to the same underlying reference, even if citation keys differ.
- Select one canonical citation key.
- Merge fields from duplicate entries instead of discarding information.
- Rewrite `\\cite{...}` commands to use the canonical key.

**Conversion method:** TODO

**Matching/canonical-key policy:** TODO

**Field union policy:** TODO

**Citation-key rewriting and tests:** TODO

### 6.5 Full-text content deduplication across versions

After cleanup, elements whose text content matches exactly across versions should share one `element_id`. The `hierarchy` table still records the parent-child edges separately for each version.

Explain:
- How content is normalized before comparison
- How IDs are generated and kept unique within a publication
- How identical content under different parents/versions is represented
- How root/document elements are handled

**Implementation:** TODO

## 7. Milestone 2 — output tables and validation

### 7.1 Expected tables

**`elements`** — one row per distinct cleaned element across all versions:
`element_id`, `arxiv_id`, `kind`, `content`

**`hierarchy`** — one row per node-parent edge for each version:
`arxiv_id`, `version`, `element_id`, `parent_id`

**`bibtex`** — unified BibTeX entries after deduplication and field merging:
`arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url`

### 7.2 Coverage and correctness checks

Report:
- Parsing success rate
- Version coverage for every publication, including empty records for versions without TeX
- Number of elements and BibTeX entries before and after deduplication
- Hierarchy integrity (every non-root parent exists, no invalid/cyclic edges)
- Citation integrity (citations use canonical keys; BibTeX keys are unique per publication)
- Cross-version element reuse after cleanup

**Validation strategy and test cases:** TODO

# TODO: Replace paths with actual output locations.
ELEMENTS_PATH = Path("data/processed/elements.parquet")
HIERARCHY_PATH = Path("data/processed/hierarchy.parquet")
BIBTEX_PATH = Path("data/processed/bibtex.parquet")

elements = load_table(ELEMENTS_PATH)
hierarchy = load_table(HIERARCHY_PATH)
bibtex = load_table(BIBTEX_PATH)

for name, frame in [
    ("elements", elements),
    ("hierarchy", hierarchy),
    ("bibtex", bibtex),
]:
    print(f"{name}: {frame.shape}")
    if not frame.empty:
        display(frame.head())

In [4]:
# TODO: Replace paths with actual output locations.
ELEMENTS_PATH = Path("data/processed/elements.parquet")
HIERARCHY_PATH = Path("data/processed/hierarchy.parquet")
BIBTEX_PATH = Path("data/processed/bibtex.parquet")

elements = load_table(ELEMENTS_PATH)
hierarchy = load_table(HIERARCHY_PATH)
bibtex = load_table(BIBTEX_PATH)

for name, frame in [
    ("elements", elements),
    ("hierarchy", hierarchy),
    ("bibtex", bibtex),
]:
    print(f"{name}: {frame.shape}")
    if not frame.empty:
        display(frame.head())

NameError: name 'Path' is not defined

### 7.3 Summary statistics and figures

Suggested report figures/tables:
- Parse success rate by version or publication
- Number of elements by kind (`sentence`, `formula`, `figure`, `heading`, etc.)
- Element count before vs. after deduplication
- BibTeX entry count before vs. after deduplication
- Distribution of hierarchy depth or number of nodes per version
- Number of empty/unparseable versions

All charts must be created from actual outputs during book compilation. Interpret the patterns and mention important caveats.

In [5]:
# TODO: Adapt after inspecting the actual schema.
if not elements.empty and "kind" in elements.columns:
    kind_counts = elements["kind"].value_counts(dropna=False)
    ax = kind_counts.plot(kind="bar")
    ax.set_title("Parsed elements by kind")
    ax.set_xlabel("Element kind")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
    

NameError: name 'elements' is not defined

## 8. Milestone 2 — performance benchmark

Measure on the CPU-only Google Colab testbed:
- End-to-end parser wall time
- Average processing time per publication
- Maximum RAM used
- Final output storage size

| Metric | Value | Measurement method / notes |
|---|---:|---|
| End-to-end parser wall time (s) | TODO | TODO |
| Average time per publication (s) | TODO | TODO |
| Maximum RAM (MiB/GiB) | TODO | TODO |
| Final output size (MiB/GiB) | TODO | TODO |

**Benchmark method:** TODO  
**Runtime configuration:** TODO


## 9. Milestone 2 — discussion and limitations

Discuss:
- Which LaTeX patterns were difficult to parse?
- How were nested sections, lists, formulas, figures, and tables handled?
- What were the main causes of parsing failures?
- Did cleanup improve cross-version deduplication?
- Were duplicate bibliography entries merged correctly and citation keys rewritten?
- What limitations remain for Milestone 3?

**Findings:** TODO

**Limitations and next steps:** TODO


# 10. Overall conclusion

Summarize what the group built in Milestones 1 and 2, the most important quality/performance findings, known limitations, and how the processed corpus will support Milestone 3.

**Conclusion:** TODO


## References

Add complete citations for documentation, libraries, APIs, and external material used in the implementation. At minimum, consult and cite the official project statements linked below.

1. Course Lab Book, *arXiv Project — Problem Statements*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/
2. Course Lab Book, *Project Milestone 1 — Data Crawling*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/
3. Course Lab Book, *Project Milestone 2 — Hierarchical Parsing and Standardization*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/
4. TODO — arXiv API / source access documentation used.
5. TODO — Semantic Scholar API documentation used.
6. TODO — libraries and other external sources used.